# 🧠 PRD-LLM Backend (Colab GPU Runner)

ဒီ Notebook က PRD-LLM ရဲ့ ဉာဏ်ရည်တု အင်ဂျင်ကို Colab GPU သုံးပြီး run ပေးမှာ ဖြစ်ပါတယ်။

### 🛠 အရေးကြီးသော Setup (မrunမီ ဒါကို အရင်လုပ်ပါ)
1. ဘယ်ဘက်ဘေးက **Key icon (Secrets)** ကို နှိပ်ပါ။
2. **Add new secret** ကို နှိပ်ပြီး `NGROK_AUTH_TOKEN` ဆိုတဲ့ နာမည်နဲ့ Token ထည့်ပါ။
3. **(Private Repo ဖြစ်လျှင်)** `GITHUB_TOKEN` ဆိုတဲ့ နာမည်နဲ့ သင်၏ [GitHub Token](https://github.com/settings/tokens) ကို ထည့်ပေးပါ။
4. Key များအားလုံး၏ ဘေးက **Notebook access ခလုတ်ကို အပြာရောင်ဖြစ်အောင် ဖွင့်ပေးပါ**။

> **သတိပြုရန်:** အပေါ်ဆုံးမှာ တက်နေတဲ့ `userdata.get('secretName')` ဆိုတဲ့ cell အမှားကို run စရာမလိုပါ။ ဖျက်ပစ်လိုက်ပါ။

In [ ]:
# [အဆင့် ၁] - Github မှ ဖိုင်များ ရယူခြင်း နှင့် Install လုပ်ခြင်း
import os
import shutil
import subprocess
from google.colab import userdata

repo_url = "https://github.com/kkomyoeminaung/Prd-llm-brain.git"
repo_name = "Prd-llm-brain"

def setup_repo():
    if os.path.exists(repo_name) and not os.path.exists(f"{repo_name}/COLAB_SERVER.py"):
        print("🧹 Cleaning incomplete folder...")
        shutil.rmtree(repo_name)

    if os.path.exists(repo_name):
        print(f"✅ {repo_name} already exists. Switching directory.")
        os.chdir(repo_name)
    else:
        print("📥 Attempting to clone repository...")
        result = subprocess.run(["git", "clone", repo_url], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ Clone successful.")
            os.chdir(repo_name)
        else:
            print("🔑 Public clone failed. Trying with GITHUB_TOKEN...")
            try:
                gh_token = userdata.get('GITHUB_TOKEN')
                auth_url = repo_url.replace("https://", f"https://{gh_token}@")
                result2 = subprocess.run(["git", "clone", auth_url], capture_output=True, text=True)
                if result2.returncode == 0:
                    print("✅ Clone with token successful.")
                    os.chdir(repo_name)
                else:
                    print(f"❌ ERROR: {result2.stderr}")
                    print("Manual နည်းလမ်း: ZIP ဒေါင်းပြီး Colab Files ထဲ Upload တင်ပါ။")
                    return
            except Exception as e:
                print(f"❌ ERROR: {e}")
                return

    print("📦 Installing requirements...")
    if os.path.exists('requirements.txt'):
        os.system("pip install -r requirements.txt -q")
    os.system("pip install pyngrok nest-asyncio -q")
    print("✅ Setup Complete!")

setup_repo()

In [ ]:
# [အဆင့် ၂] - Server စတင်ခြင်း
from google.colab import userdata
import os
import sys

print("🔍 Validating environment...")
try:
    # Ngrok Token
    token = userdata.get('NGROK_AUTH_TOKEN')
    os.environ['NGROK_AUTH_TOKEN'] = token
    
    # လက်ရှိ folder ထဲမှာ server file ရှိမရှိ စစ်မယ်
    if not os.path.exists('COLAB_SERVER.py'):
        print("⚠️ COLAB_SERVER.py not found in current folder. Trying to find it...")
        if os.path.exists('/content/Prd-llm-brain/COLAB_SERVER.py'):
            %cd /content/Prd-llm-brain
    
    if os.path.exists('COLAB_SERVER.py'):
        print("🚀 Launching PRD-LLM Engine on Colab GPU...")
        !python COLAB_SERVER.py
    else:
        print("❌ CRITICAL ERROR: COLAB_SERVER.py မရှိပါ။ Repository clone လုပ်တာ မအောင်မြင်ခဲ့တာ ဖြစ်နိုင်ပါတယ်။")
        !ls -R
except Exception as e:
    print(f"❌ ERROR: {e}")
    print("Tips: Secrets (Key icon) မှာ NGROK_AUTH_TOKEN ကို အမှန်ထည့်ပြီး Access ဖွင့်ထားဖို့ သေချာပါစေ။")